## Assignment III: Growth Rates, Subplot Grids, and 3D Surfaces

**Total: 50 points.** Question 1 is 20 points, Questions 2 and 3 are 15 points each.

This assignment pulls together the material on polynomial vs. exponential growth (from *Limits and Convergence of Algorithms*) and plotting with Matplotlib (from *Intro to Python V*): `loglog` and `semilogy` axes, fitting rates with `np.polyfit`, grids of subplots with shared axes, and 3D surfaces built with `np.meshgrid`.

- Each question is split into **Part (a)** and **Part (b)**.
- Write your code in the cell provided for each part. Make sure it **prints** your numerical answers clearly (ideally with an f-string that says what the printed value represents) and **shows** every requested figure.
- Some parts ask a written question. Answer those in the **markdown cell** provided right below the code cell.
- Every figure must have axis labels and a title. Panels that show more than one curve need a legend.
- **Submission:** see the instructions at the very end of this notebook.

### AI Usage Policy

**No AI tools (ChatGPT, Claude, GitHub Copilot's suggestion/chat features, etc.) may be used to generate, complete, or rewrite your solutions for this assignment.** All code and explanations must be your own work.

**Exception and disclosure requirement:** If you have **GitHub Copilot active** in your editor (e.g. inline autocomplete suggestions appearing as you type), you do not need to disable it, but you **must disclose this** by adding a short **markdown cell right below this one** stating:
- That Copilot (or any other AI tool) was active while you worked, and
- Briefly, how you used or did not use its suggestions (e.g. "Copilot was active but I did not accept any of its suggestions for graded cells" or "I accepted a Copilot suggestion for the `ax.set_xlabel` lines in Q2(a) but wrote the rest myself").

Notebooks with AI-generated solutions that are not disclosed as above will not receive credit, regardless of correctness.

**AI Usage Disclosure (student to fill in):**

*(If GitHub Copilot or any other AI tool was active while you worked on this assignment, describe that here. If no AI tool was active at all, write "No AI tools were used.")*

Run this cell first (do not change it).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Question 1: Polynomial or Exponential? (20 points)

Below are two algorithms. Instead of timing them with a clock (which is noisy and depends on your computer), each function **counts its own steps** and returns that count. Your job is to decide, from data, whether the number of steps grows **polynomially**, $t \approx c\,n^a$, or **exponentially**, $t \approx c\,b^{\,n}$, and to estimate the rate ($a$ or $b$).

- **`algorithm_A(n)`** is brute-force **3SUM**: given $n$ integers, it checks every triple of positions $i < j < k$ to see whether the three numbers add up to zero. One step = one triple checked.
- **`algorithm_B(n)`** computes the Fibonacci number $F_n$ with the **naive recursion** $F_n = F_{n-1} + F_{n-2}$ (with $F_0 = 0$, $F_1 = 1$). One step = one call of the function `fib_naive`.

Run the cell below to define them (do not change it).

In [ ]:
def algorithm_A(n):
    '''brute-force 3SUM on n random integers; returns the number of triples checked'''
    rng = np.random.default_rng(seed=n)
    a = rng.integers(-100, 101, size=n).tolist()    # n random integers between -100 and 100
    steps = 0
    zero_triples = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                steps += 1                          # one triple (i, j, k) checked
                if a[i] + a[j] + a[k] == 0:
                    zero_triples += 1
    return steps


def fib_naive(n, counter):
    '''naive recursive Fibonacci; counter is a one-element list that counts the calls'''
    counter[0] += 1                                 # this call is one step
    if n < 2:
        return n
    return fib_naive(n - 1, counter) + fib_naive(n - 2, counter)


def algorithm_B(n):
    '''returns the number of calls fib_naive makes to compute F_n'''
    counter = [0]
    fib_naive(n, counter)
    return counter[0]


ns_A = np.arange(10, 170, 10)    # n = 10, 20, ..., 160   (16 values)
ns_B = np.arange(5, 25)          # n = 5, 6, ..., 24      (20 values)

**(a) [8 pts] Look before you fit.**
1. Build NumPy arrays `steps_A` (the value of `algorithm_A(n)` for every `n` in `ns_A`) and `steps_B` (the value of `algorithm_B(n)` for every `n` in `ns_B`). Print both arrays.
2. Make **one figure with two panels side by side** (a $1 \times 2$ grid, e.g. `figsize=(11, 4)`):
   - **left panel:** both data sets on `loglog` axes;
   - **right panel:** both data sets on `semilogy` axes.

   Plot each data set with markers (e.g. `"o-"`) and a label (`"A"` and `"B"`). Every panel needs an x label, a y label, a title saying which kind of axes it uses, and a legend.
3. In the markdown cell below the code, state which data set looks like a straight line on which axes, and your resulting guess: is A polynomial or exponential? Is B?

In [ ]:
# TODO: your code for Q1(a) here

**Your answer for Q1(a):**

*(Which data set is straight on which axes? Your guess for A and for B.)*

**(b) [12 pts] Fit the rates and confirm with the split test.**
1. Write (or reuse from Activity XIV) `fit_loglog(ns, ts)`, which returns the slope of $\log t$ against $\log n$, and `fit_semilogy(ns, ts)`, which returns $b = e^{\text{slope}}$, where the slope is that of $\log t$ against $n$.
2. Write (or reuse from Activity XIV) `split_test(fit, ns, ts)`, which returns `[fit(first half), fit(second half)]`.
3. For **each** of A and B, print the split test with **both** fitting functions (four results in total). Use the results to confirm or correct your guesses from part (a).
4. Using the **correct** model only, print your estimated rate for each algorithm, fitted on **all** of its data: the exponent $a$ for the polynomial one and the base $b$ for the exponential one.
5. **Compare with theory.** Print your estimates next to the exact values:
   - The number of triples $i < j < k$ chosen from $n$ positions is $\binom{n}{3} = \dfrac{n(n-1)(n-2)}{6}$, which behaves like $\dfrac{n^3}{6}$ for large $n$. So the true exponent is $a = 3$.
   - The number of calls $C_n$ satisfies $C_n = 1 + C_{n-1} + C_{n-2}$, almost the Fibonacci recursion itself, so it grows like the golden ratio: the true base is $\varphi = \dfrac{1 + \sqrt{5}}{2} \approx 1.618$.

   Print the relative error $\dfrac{|\text{estimate} - \text{true}|}{\text{true}}$ for both.

In the markdown cell below, explain in two or three sentences how the split test told you which model is right. Also explain why the polynomial fit on the **second** half is closer to $3$ than the fit on the first half. (*Hint:* $n(n-1)(n-2)$ versus $n^3$ when $n$ is small and when $n$ is large.)

In [ ]:
# TODO: your code for Q1(b) here

**Your answer for Q1(b):**

*(How did the split test decide? Why is the second-half exponent closer to 3?)*

### Question 2: A $3 \times 2$ Grid of Functions (15 points)

Here are six functions, all plotted on the same interval $x \in [-2\pi, 2\pi]$:

| panel | function | name |
|---|---|---|
| row 0, left | $f_1(x) = \sin x$ | sine |
| row 0, right | $f_2(x) = \cos x$ | cosine |
| row 1, left | $f_3(x) = \dfrac{\sin x}{x}$ | sinc |
| row 1, right | $f_4(x) = \tanh x$ | hyperbolic tangent |
| row 2, left | $f_5(x) = e^{-x^2/8}\cos(3x)$ | wave packet |
| row 2, right | $f_6(x) = \dfrac{4}{\pi}\left(\sin x + \dfrac{\sin 3x}{3} + \dfrac{\sin 5x}{5}\right)$ | Fourier sum for a square wave |

**Note on $f_3$:** computing `np.sin(x) / x` divides by zero if one of your `x` values is exactly `0`. Use `np.sinc(x / np.pi)` instead: NumPy's `np.sinc(t)` is defined as $\sin(\pi t)/(\pi t)$, so `np.sinc(x / np.pi)` equals $\sin(x)/x$, and it correctly returns $1$ at $x = 0$ (the limit $\lim_{x \to 0} \sin(x)/x = 1$).

**(a) [10 pts] The figure.** Create `x = np.linspace(-2 * np.pi, 2 * np.pi, 500)`. Make one figure with a $3 \times 2$ grid of panels using
```python
fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(10, 9), sharex=True, sharey=True)
```
so all six panels **share the x-axis and the y-axis**. Then:
1. Plot each function in its own panel, in the positions given in the table. `axs[i, j]` is the panel in row `i`, column `j` (counting from `0`).
2. Give each panel a title showing its formula in LaTeX, e.g. `axs[0, 0].set_title(r"$\sin x$")`.
3. Put an **x label** (`"x"`) on the **bottom row only** and a **y label** (e.g. `"f(x)"`) on the **left column only**. Since the axes are shared, labels on the inner panels would just repeat the same information.
4. Draw a thin horizontal line at $y = 0$ in every panel (`ax.axhline(0, color="gray", lw=0.8)`), add one overall title with `fig.suptitle(...)`, and finish with `plt.tight_layout()`.

**(b) [5 pts] Reading the figure.** Because the axes are shared, you can compare the panels directly. In code, print the **maximum value** of $f_6$ on your grid `x`, using `np.max`. Then, in the markdown cell below the code:
1. List which of the six functions are **even** ($f(-x) = f(x)$) and which are **odd** ($f(-x) = -f(x)$), and say how you can see this in the picture.
2. The square wave that $f_6$ approximates only takes the values $\pm 1$, yet the maximum you printed is larger than $1$. In one sentence, describe where in the panel this overshoot happens. (This overshoot near a jump is called the **Gibbs phenomenon**, and it does not go away when more terms are added to the sum.)
3. In one sentence: why does `sharey=True` make it fair to compare the heights of the six curves?

In [ ]:
# TODO: your code for Q2(a) here

In [ ]:
# TODO: your code for Q2(b) here

**Your answer for Q2(b):**

*(Even and odd functions; where the overshoot of $f_6$ happens; why `sharey=True` helps.)*

### Question 3: Surfaces in 3D with `np.meshgrid` (15 points)

Recall: for 1D arrays `x` (length $m$) and `y` (length $n$), `X, Y = np.meshgrid(x, y)` returns two 2D arrays of shape `(n, m)` with `X[i, j] = x[j]` and `Y[i, j] = y[i]`. Any formula written with `X` and `Y` is then evaluated at **every** grid point at once, with no loops. To draw a surface, use a 3D Axes and `plot_surface`:
```python
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(1, 1, 1, projection="3d")
surf = ax.plot_surface(X, Y, Z, cmap="viridis")
```

**(a) [8 pts] The saddle.**
1. Create `x = np.linspace(-2, 2, 81)` and `y = np.linspace(-2, 2, 61)`, and build `X, Y` with `np.meshgrid`. Print `X.shape` and `Y.shape`, and confirm in a comment which of the two lengths, $81$ or $61$, gives the number of rows.
2. Compute $Z = X^2 - Y^2$ and plot it with `plot_surface` and a colormap of your choice.
3. Label all three axes (`ax.set_xlabel`, `ax.set_ylabel`, `ax.set_zlabel`), give the plot a title with the formula in LaTeX, add a colorbar (`fig.colorbar(surf, ax=ax, shrink=0.6)`), and choose a viewing angle with `ax.view_init(elev=..., azim=...)` that clearly shows the saddle shape.
4. In a comment or a print statement: the origin $(0, 0)$ is a critical point of $f(x, y) = x^2 - y^2$. Along which axis does the surface curve **up** away from the origin, and along which does it curve **down**? (This is exactly why $(0, 0)$ is a saddle point and neither a minimum nor a maximum.)

**(b) [7 pts] The monkey saddle and complex numbers.** For a complex number $z = x + iy$, the real and imaginary parts of $z^3$ are
$$
\mathrm{Re}(z^3) = x^3 - 3xy^2, \qquad \mathrm{Im}(z^3) = 3x^2 y - y^3 .
$$
The surface $u = x^3 - 3xy^2$ is called the **monkey saddle**: it has three "valleys" (two for the legs, one for the tail).
1. Using the same `X` and `Y` from part (a), compute `W = (X + 1j * Y)**3`, a 2D array of complex numbers (complex arithmetic works elementwise on arrays, just like real arithmetic). Set `U = W.real` and `V = W.imag`. Use `np.allclose` to confirm that `U` equals $X^3 - 3XY^2$ and that `V` equals $3X^2Y - Y^3$.
2. Instead of a 3D surface, show $u$ and $v$ as two **density plots** (filled contour plots, `contourf`) side by side: $u = \mathrm{Re}(z^3)$ on the left and $v = \mathrm{Im}(z^3)$ on the right. The color at each point $(x, y)$ shows the height of the function there, like a map seen from above. The code cell below already contains a **starting point**: the left panel is done for you. Complete the right panel in the same way (plot `V`, add its colorbar, labels and a title), then add one overall title with `fig.suptitle(...)`. Keep the shared `levels` and the `set_aspect("equal")` line, so the two panels use the same color scale and circles look round.
3. In the markdown cell below, answer: both density plots show the same pattern of three red sectors and three blue sectors around the origin, but turned about the origin relative to each other. By what angle is the pattern of $v$ turned relative to that of $u$, and in which direction? (The square plotting window does not turn with the pattern, so the corners look different; compare the sectors near the origin.) (*Hint:* in polar coordinates $z = re^{i\theta}$, so $z^3 = r^3 e^{3i\theta}$, which gives $u = r^3\cos 3\theta$ and $v = r^3 \sin 3\theta$. Use $\sin \alpha = \cos(\alpha - \pi/2)$.)

In [ ]:
# TODO: your code for Q3(a) here

In [ ]:
# TODO: your code for Q3(b) here
# Step 1: compute W, U, V and check them with np.allclose (your code)


# Step 2: density plots (starting point: the left panel is done, complete the right panel)
levels = np.linspace(-16, 16, 33)              # the same color levels for both panels
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

cf0 = axs[0].contourf(X, Y, U, levels=levels, cmap="coolwarm")   # filled contour ("density") plot of U
fig.colorbar(cf0, ax=axs[0], label="u")
axs[0].set_xlabel("x")
axs[0].set_ylabel("y")
axs[0].set_title(r"$u = \mathrm{Re}(z^3) = x^3 - 3xy^2$")
axs[0].set_aspect("equal")                     # one unit in x has the same length as one unit in y

# TODO: the right panel axs[1], showing V in the same way


# TODO: an overall title with fig.suptitle(...)

plt.tight_layout()
plt.show()

**Your answer for Q3(b):**

*(By what angle is the surface of $v$ turned relative to the surface of $u$? Show the short calculation.)*

### Submission Instructions

**Submit this assignment on Canvas only.** There is no GitHub submission for this assignment.

1. Before submitting, choose **Kernel → Restart & Run All** (or **Restart Kernel and Run All Cells**) and check that every cell runs without errors and that **all figures are visible** in the notebook. Figures that do not appear in the submitted notebook cannot be graded.
2. Make sure your written answers are filled in, in the markdown cells marked *Your answer*, and that the AI Usage Disclosure at the top is filled in.
3. Upload the completed `.ipynb` file to the Assignment III page on Canvas.

No late submissions will be accepted.